### Middelware

1. tracking agent's behaviour => with loging , analytics, debugging ]]
2. applying gardrails, Pll detection. 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [ ]:

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
import os
from dotenv import load_dotenv
load_dotenv()
import langchain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
langchain.__version__

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

# model = init_chat_model("gemini-flash", model_provider="google_genai")

llm = ChatGoogleGenerativeAI(
    model="groq:qwen/qwen3-32b",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",4),
            keep=("messages",2)
        )
    ]
)


In [25]:
## Run with thread id 

config={"configurable": {"thread_id":"test-1"}}

In [26]:
## Alternativetest data 
questions = [
    "what is 2+2?",
    "what is 100/4?",
    "what is 10*5?",
    "what is 15-7?",
    "what is 3*3?",
    "what is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"messages: {response}")
    print(f"messages: {len(response['messages'])}")

messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='0e76fdf4-d347-43e9-8f05-afe648c6e05f'), AIMessage(content='<think>\nOkay, the user is asking "what is 2+2?" Hmm, that\'s a basic arithmetic question. Let me think. In standard mathematics, 2 plus 2 equals 4. But wait, maybe they\'re looking for a different interpretation? Like in some contexts, people joke about 2+2 being 5, but that\'s usually in political or satirical contexts. Or maybe in some alternative math systems? But the user probably wants the straightforward answer. Let me confirm if there\'s any trick here. The question seems simple, so unless there\'s a specific context mentioned, the answer is 4. I should also consider if they want a detailed explanation, but since it\'s so basic, maybe just state the answer clearly. Yeah, I\'ll go with 4.\n</think>\n\nThe sum of 2 and 2 is **4**. \n\nIn standard arithmetic, $ 2 + 2 = 4 $. Let me know if you\'d like further clarifi

## Token Size



In [27]:
from langchain.chat_models import init_chat_model
from langgraph.types import Checkpointer
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

@tool
def search_hotels(city: str) -> str:
    """Search Hotels - returns ;ong response to use more tokens."""
    return f"""Hotels in {city}
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City inn - 4 star , $180/night , buisness center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer= InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("tokens",550),
            keep=("tokens",200),
        )
    ]
)

config={"configurable": {"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars= sum(len(str(m.content)) for m in messages)
    return total_chars // 4 

In [28]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~138 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='a2b83134-5bba-4cfd-b5ed-b5f2c33f85a6'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. The required parameter is city, and it\'s a string. Since the user mentioned Paris, I need to call this function with "Paris" as the city argument. I\'ll make sure the JSON is correctly formatted with the city name in quotes. No other parameters are needed here. Alright, time to structure the tool call properly.\n', 'tool_calls': [{'id': 't99qg858m', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 156, 'total_tokens': 279, 'completion_time': 0.190654957, 'completion_tokens_details': {'reasoni

KeyboardInterrupt: 

In [29]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~65 tokens (0.0508%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='dad1f79b-fc8e-4465-bdcc-a4a319e9cd7c'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for hotels in Paris. Let me check the tools available. There's a function called search_hotels that takes a city parameter. Since the user specified Paris, I need to call this function with the city set to Paris. I'll make sure the arguments are correctly formatted as JSON within the tool_call tags.\n", 'tool_calls': [{'id': 'b66jr0h67', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 147, 'total_tokens': 240, 'completion_time': 0.142223129, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.006005668, 'prompt_tokens_details': None, 'queue_time': 0.052942349, 'total_time': 0.148228797}, 'model_n

In [31]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [32]:
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)



In [ ]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [34]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='8173d15a-8b55-404b-80ce-2fdfc1749cf9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three pieces of information: recipient is john@test.com, subject is Hello, and body is How are you? So I need to call the send_email_tool with those arguments. I'll structure the JSON accordingly, making sure to include each parameter correctly. No need to use the read_email_tool here since the task is about sending, not reading. Everything seems in order. Time to format the tool_call.\n", 'tool_calls': [{'id': '9psdgqra2', 'function': {'argu

In [35]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: ✅ Successfully sent email to **john@test.com** with subject **"Hello"**. Let me know if you need further assistance!


In [ ]:
result

In [36]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [37]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: It seems there was an issue processing your request to send the email. The system rejected the tool call with ID `gce78wznm`. This could be due to a temporary error or an issue with the mock function. Please try again, or verify the email details (e.g., valid recipient, subject, and body). If the problem persists, contact support for further assistance.


In [38]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='ff64705e-4a17-480d-a6d6-e7a1b6892204'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are required. The parameters are provided in the user's request, so I can use that function. I need to structure the JSON with those parameters. Let me make sure the keys are correct: recipient, subject, body. Yes, that's right. So the tool call should include all three. No need for the read_email_tool here since the user isn't asking to read an email. Just send it using the send_email_tool.\n", 'tool_calls': [{'id': 'gce78wznm', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subj